
# 00 - Setup, API verification and the distance diagnostic

**Run this notebook first.** Nothing else in the project should be trusted until it passes.

It does four things:

1. **Verifies the PyG API surface** the other 30 notebooks depend on, so an API change
   surfaces here in two minutes rather than three hours into a training run.
2. **Reports dataset facts** (`N_CLASSES`, `NODE_VOCAB`, feature widths) that the leaf
   notebooks hard-code. If a number here disagrees with a leaf notebook, the leaf notebook
   is wrong.
3. **Runs the distance diagnostic** - the histogram of hop distances and the edge blow-up
   factor at each radius `r`. This is a *decision gate*: it tells us whether the traversal
   encoding carries real information on each dataset, or whether the graph is so dense that
   every pair sits at distance 1 or 2 and the signal degenerates.
4. **Tests correctness**: the traversal encoding against a textbook BFS, and the
   permutation-invariance of the whole pipeline.

In [ ]:
!pip -q install torch-geometric

In [ ]:

import os, time, math, itertools
from collections import deque, Counter
import numpy as np
import torch
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt

import torch_geometric
from torch_geometric.datasets import GNNBenchmarkDataset

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RESULT_ROOT, DATA_ROOT = "/content/drive/MyDrive/trpe", "/content/trpe_data"
except Exception:
    RESULT_ROOT, DATA_ROOT = os.path.abspath("./trpe"), os.path.abspath("./trpe_data")

FIG_DIR = os.path.join(RESULT_ROOT, "figures")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(DATA_ROOT, exist_ok=True)

print("torch          ", torch.__version__)
print("torch_geometric", torch_geometric.__version__)
print("cuda           ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY")
print("data ->", DATA_ROOT)
print("figs ->", FIG_DIR)


## 1. API verification

The leaf notebooks assume four things about PyG. Each is checked below; a `FAIL` here
means the corresponding leaf notebooks need editing before they are run.

In [ ]:

checks = {}

# (a) GNNBenchmarkDataset serves CLUSTER and CIFAR10 with predefined splits.
for name in ["CLUSTER", "CIFAR10"]:
    try:
        d = GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name, split="val")
        checks[f"{name} split='val'"] = f"OK  ({len(d)} graphs)"
    except Exception as e:
        checks[f"{name} split='val'"] = f"FAIL {type(e).__name__}: {e}"

# (b) CSL has no predefined split - confirm it loads as one collection, which is why the
#     CSL notebooks use stratified k-fold instead.
try:
    csl = GNNBenchmarkDataset(os.path.join(DATA_ROOT, "CSL"), "CSL")
    checks["CSL (no split arg)"] = f"OK  ({len(csl)} graphs)"
except Exception as e:
    checks["CSL (no split arg)"] = f"FAIL {type(e).__name__}: {e}"
try:
    GNNBenchmarkDataset(os.path.join(DATA_ROOT, "CSL"), "CSL", split="train")
    checks["CSL split='train'"] = "UNEXPECTEDLY OK - a predefined split exists, prefer it over k-fold"
except Exception as e:
    checks["CSL split='train'"] = f"absent as expected ({type(e).__name__}) -> k-fold is correct"

# (c) The two baseline PE transforms exist under the names the leaf notebooks import.
try:
    from torch_geometric.transforms import AddLaplacianEigenvectorPE, AddRandomWalkPE
    checks["AddLaplacianEigenvectorPE / AddRandomWalkPE"] = "OK"
except Exception as e:
    checks["AddLaplacianEigenvectorPE / AddRandomWalkPE"] = f"FAIL {e}"

# (d) GATConv accepts edge features - this is how our relative bias reaches the attention
#     logits, so without it the GAT half of the project has no mechanism.
try:
    from torch_geometric.nn import GATConv
    conv = GATConv(8, 4, heads=2, edge_dim=5)
    out = conv(torch.randn(6, 8), torch.tensor([[0, 1, 2], [1, 2, 3]]), edge_attr=torch.randn(3, 5))
    checks["GATConv(edge_dim=...)"] = f"OK  (output {tuple(out.shape)})"
except Exception as e:
    checks["GATConv(edge_dim=...)"] = f"FAIL {type(e).__name__}: {e}"

w = max(len(k) for k in checks)
for k, v in checks.items():
    print(f"{k:<{w}}  {v}")


## 2. Dataset facts

These are the constants the leaf notebooks hard-code (`N_CLASSES`, `NODE_VOCAB`, and the
input feature width). Hard-coding them avoids an expensive full-dataset scan at the start
of every one of the 30 runs; verifying them here is the price of that.

In [ ]:

def base_features(data, dataset, node_vocab):
    """Identical to the feature preparation in the leaf notebooks."""
    if dataset in ("CIFAR10", "MNIST") and getattr(data, "pos", None) is not None:
        data.x = torch.cat([data.x.float(), data.pos.float()], dim=-1)
    if data.x is None:
        data.x = torch.ones((data.num_nodes, 1), dtype=torch.float)
    if data.x.dim() == 1:
        data.x = F.one_hot(data.x.long().clamp(0, node_vocab - 1), node_vocab).float()
    return data.x.float()


def dataset_facts(name, split=None, sample=400):
    ds = (GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name, split=split) if split
          else GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name))
    idx = np.linspace(0, len(ds) - 1, min(sample, len(ds))).astype(int)
    n_nodes, n_edges, ys, xmax, xdim = [], [], [], -1, None
    for i in idx:
        d = ds[int(i)]
        n_nodes.append(d.num_nodes); n_edges.append(d.edge_index.size(1))
        ys.append(d.y.view(-1))
        if d.x is not None:
            xdim = "discrete node type" if d.x.dim() == 1 else d.x.size(1)
            if d.x.dim() == 1:
                xmax = max(xmax, int(d.x.max()))
        else:
            xdim = "none (featureless)"
    y = torch.cat(ys)
    return dict(
        dataset=name, graphs=len(ds), avg_nodes=float(np.mean(n_nodes)),
        avg_edges=float(np.mean(n_edges)),
        avg_degree=float(np.mean(n_edges) / np.mean(n_nodes)),
        raw_x=xdim, node_vocab=(xmax + 1) if xmax >= 0 else 0,
        n_classes=int(y.max()) + 1,
        task="node" if y.numel() > len(idx) else "graph",
    )


facts = [dataset_facts("CSL"), dataset_facts("CLUSTER", "train"), dataset_facts("CIFAR10", "train")]
hdr = f"{'dataset':<10}{'graphs':>8}{'avg n':>9}{'avg m':>10}{'avg deg':>9}{'classes':>9}{'vocab':>7}  {'raw x':<20}{'task':<7}"
print(hdr); print("-" * len(hdr))
for f in facts:
    print(f"{f['dataset']:<10}{f['graphs']:>8}{f['avg_nodes']:>9.1f}{f['avg_edges']:>10.1f}"
          f"{f['avg_degree']:>9.1f}{f['n_classes']:>9}{f['node_vocab']:>7}  {str(f['raw_x']):<20}{f['task']:<7}")

print("\nCross-check against the leaf notebooks (N_CLASSES / NODE_VOCAB / TASK):")
expected = {"CSL": (10, 0, "graph"), "CLUSTER": (6, 7, "node"), "CIFAR10": (10, 0, "graph")}
for f in facts:
    e = expected[f["dataset"]]
    got = (f["n_classes"], f["node_vocab"], f["task"])
    print(f"  {f['dataset']:<9} expected {e}  got {got}  {'OK' if e == got else '<<< MISMATCH, fix the leaf notebooks'}")


## 3. The distance diagnostic

This is the decision gate for the whole experimental design.

The traversal encoding labels each rewired edge with its hop distance $d(u,v)\in\{1..r\}$.
That label is only informative if the distances are actually *spread out*. On a graph with
average degree $\approx 37$ and diameter 2 - which is what an SBM like CLUSTER looks like -
every pair sits at distance 1 or 2, the $r$-hop rewiring nearly completes the graph, and the
"relative position" of a node pair carries almost no information.

So for each dataset we measure:

* the **distribution of hop distances** in the rewired graph,
* the **edge blow-up factor** $|E_r|/|E_1|$, which is also the training-cost multiplier,
* the **diameter**, which bounds how much signal any distance-based encoding can carry.

The expectation going in: CIFAR10-SP (an 8-nearest-neighbour superpixel graph) is sparse
with a large diameter and should show a rich distance profile; CLUSTER should be nearly
degenerate. If that holds, the two datasets form a deliberate sparse-vs-dense contrast
rather than two attempts at the same experiment.

In [ ]:

def trpe_arrays(edge_index, n, r):
    """Same algebra as the leaf notebooks: (A^k)>0 first at k = d(u,v)."""
    ei = edge_index.numpy()
    A = np.zeros((n, n)); A[ei[0], ei[1]] = 1.0
    A = np.maximum(A, A.T); np.fill_diagonal(A, 0.0)
    dist = np.full((n, n), -1, dtype=np.int64); np.fill_diagonal(dist, 0)
    nsp = np.zeros((n, n))
    Mk = np.eye(n)
    for k in range(1, r + 1):
        Mk = Mk @ A
        fresh = (Mk > 0.5) & (dist < 0)
        if not fresh.any(): break
        dist[fresh] = k; nsp[fresh] = Mk[fresh]
    return dist, nsp


def diagnose(name, split=None, sample=150, rmax=4):
    ds = (GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name, split=split) if split
          else GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name))
    idx = np.linspace(0, len(ds) - 1, min(sample, len(ds))).astype(int)
    hist, diam, base_edges = Counter(), [], 0
    for i in idx:
        d = ds[int(i)]
        n = int(d.num_nodes)
        dist, _ = trpe_arrays(d.edge_index, n, n)      # r = n  =>  full distance matrix
        base_edges += d.edge_index.size(1)
        vals = dist[dist >= 1]
        hist.update(vals.tolist())
        diam.append(int(vals.max()) if vals.size else 0)

    tot = sum(hist.values())
    e1 = hist.get(1, 1)
    print(f"\n=== {name} ({len(idx)} graphs sampled) ===")
    print(f"  mean diameter {np.mean(diam):.1f}   max diameter {max(diam)}")
    print(f"  {'r':>3} {'share of pairs at d=r':>23} {'cumulative':>12} {'|E_r|/|E_1|':>13}")
    cum = 0
    for r in range(1, min(rmax, max(hist)) + 1):
        c = hist.get(r, 0); cum += c
        print(f"  {r:>3} {100*c/tot:>22.1f}% {100*cum/tot:>11.1f}% {cum/e1:>13.1f}x")
    frac_le2 = (hist.get(1, 0) + hist.get(2, 0)) / tot
    verdict = ("DEGENERATE - almost every pair is at distance 1 or 2, so the relative "
               "signal is nearly binary" if frac_le2 > 0.95 else
               "INFORMATIVE - distances are well spread")
    print(f"  verdict: {verdict}")
    return name, hist, np.mean(diam)


diags = [diagnose("CSL"), diagnose("CLUSTER", "val"), diagnose("CIFAR10", "val")]

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (name, hist, mdiam) in zip(axes, diags):
    tot = sum(hist.values())
    ks = sorted(hist)[:12]
    ax.bar(ks, [100 * hist[k] / tot for k in ks], color="#3b6ea5")
    ax.set_title(f"{name}  (mean diam {mdiam:.1f})")
    ax.set_xlabel("hop distance $d(u,v)$"); ax.set_ylabel("% of node pairs")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Distance profile: how much information a relative encoding can carry", y=1.04)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig_distance_profile.pdf"), bbox_inches="tight")
plt.show()
print("saved ->", os.path.join(FIG_DIR, "fig_distance_profile.pdf"))


## 4. Correctness of the traversal encoding

The leaf notebooks compute hop distances and shortest-path counts algebraically rather than
with an explicit BFS, because $O(n)$ Python-level traversals per graph is far too slow for
45k graphs. The equivalence is not obvious, so it is tested rather than asserted:

* $(A^k)_{uv}$ counts **walks** of length $k$, so the first $k$ with $(A^k)_{uv}>0$ is
  $d(u,v)$;
* at $k=d(u,v)$ every length-$k$ walk is necessarily a shortest **path**, so $(A^{d})_{uv}$
  is the number of shortest paths.

Both claims are checked below against a textbook BFS with shortest-path counting.

In [ ]:

def reference_bfs(A, r):
    """Textbook BFS with shortest-path counting, one source at a time."""
    n = A.shape[0]
    adj = [np.nonzero(A[i])[0].tolist() for i in range(n)]
    dist = np.full((n, n), -1, dtype=np.int64); nsp = np.zeros((n, n))
    for s in range(n):
        dist[s, s] = 0; nsp[s, s] = 1
        q = deque([s])
        while q:
            x = q.popleft()
            if dist[s, x] >= r: continue
            for y in adj[x]:
                if dist[s, y] == -1:
                    dist[s, y] = dist[s, x] + 1; nsp[s, y] = nsp[s, x]; q.append(y)
                elif dist[s, y] == dist[s, x] + 1:
                    nsp[s, y] += nsp[s, x]
    np.fill_diagonal(nsp, 0.0)
    return dist, nsp


def algebraic(A, r):
    n = A.shape[0]
    dist = np.full((n, n), -1, dtype=np.int64); np.fill_diagonal(dist, 0)
    nsp = np.zeros((n, n)); Mk = np.eye(n)
    for k in range(1, r + 1):
        Mk = Mk @ A
        fresh = (Mk > 0.5) & (dist < 0)
        if not fresh.any(): break
        dist[fresh] = k; nsp[fresh] = Mk[fresh]
    return dist, nsp


rng = np.random.RandomState(0)
for trial in range(300):
    n = rng.randint(4, 30); p = rng.uniform(0.08, 0.5)
    A = (rng.rand(n, n) < p).astype(float); A = np.maximum(A, A.T); np.fill_diagonal(A, 0)
    for r in (1, 2, 3, 4):
        d1, s1 = algebraic(A, r); d2, s2 = reference_bfs(A, r)
        assert (d1 == d2).all(), f"distance mismatch (n={n}, r={r})"
        m = d1 >= 1
        assert np.allclose(s1[m], s2[m]), f"shortest-path-count mismatch (n={n}, r={r})"
print("PASS  300 random graphs x r in {1,2,3,4}")
print("      algebraic hop distances and shortest-path counts match reference BFS exactly")


## 5. Permutation invariance

This is the property that makes the encoding well defined at all. A traversal encoding built
from BFS **visit order** - the index at which a node is popped from the queue - depends on
the order in which neighbours happen to be stored, so relabelling the nodes changes the
encoding and the model is not a function of the graph. A traversal encoding built from BFS
**distance** does not have that problem.

The test below relabels each graph by a random permutation, recomputes the encoding, and
checks that the resulting multiset of `(distance, shortest-path count)` labels is unchanged -
and, for contrast, that a visit-order encoding fails the same test.

In [ ]:

def dist_signature(A, r):
    d, s = algebraic(A, r)
    m = d >= 1
    return sorted(zip(d[m].tolist(), np.round(s[m], 6).tolist()))


def visit_order_signature(A, r):
    """A BFS *visit-order* encoding, for contrast. Root = lowest-index max-degree node."""
    n = A.shape[0]
    adj = [np.nonzero(A[i])[0].tolist() for i in range(n)]
    root = int(np.argmax(A.sum(1)))
    order, seen, q = [], {root}, deque([root])
    while q:
        x = q.popleft(); order.append(x)
        for y in adj[x]:
            if y not in seen:
                seen.add(y); q.append(y)
    rank = np.zeros(n)
    for i, v in enumerate(order):
        rank[v] = i / max(1, len(order))
    return sorted(np.round(rank, 6).tolist())


rng = np.random.RandomState(1)
dist_ok = order_ok = 0
TRIALS = 200
for _ in range(TRIALS):
    n = rng.randint(6, 25)
    A = (rng.rand(n, n) < rng.uniform(0.15, 0.4)).astype(float)
    A = np.maximum(A, A.T); np.fill_diagonal(A, 0)
    P = rng.permutation(n)
    B = A[np.ix_(P, P)]
    dist_ok  += dist_signature(A, 3) == dist_signature(B, 3)
    order_ok += visit_order_signature(A, 3) == visit_order_signature(B, 3)

print(f"BFS-DISTANCE encoding (ours)  : invariant on {dist_ok}/{TRIALS} relabellings")
print(f"BFS-VISIT-ORDER encoding      : invariant on {order_ok}/{TRIALS} relabellings")
assert dist_ok == TRIALS, "our encoding must be permutation invariant"
print("\nThe distance-based encoding is a function of the graph; the order-based one is not.")


## 6. What this notebook decided

Read the three outputs above together before running any leaf notebook:

* **API check** - all four lines must read `OK`. `GATConv(edge_dim=...)` is the critical one:
  without it the relative bias has no route into the attention logits and the entire GAT
  half of the project has no mechanism.
* **Dataset facts** - every line must read `OK`. A mismatch means a leaf notebook has a wrong
  `N_CLASSES` or `NODE_VOCAB` and will train against a silently malformed target.
* **Distance diagnostic** - this fixes the radius `r` per dataset and, if CLUSTER comes back
  `DEGENERATE`, it is reported in the paper as the dense-graph regime rather than quietly
  dropped. The edge blow-up column is also the training-cost multiplier for M3/M4, so it is
  what the runtime estimates in the leaf notebooks are based on.

Next: `01_expressivity.ipynb` (minutes, no training), then the leaf notebooks under
`gcn/` and `gat/`.